# J/$\psi$ PID Efficiency

In [ ]:
# This code gets the output from O2 task PIDefficiencyConverter.cxx, which measures weighted J/psi PID efficiency from electron PID efficiency maps, and normalizes it

In [ ]:
import ROOT
import utils.hist_utils as hutils # functions created by me
from importlib import reload
import numpy as np
hutils.test_utils()

---
# Inclusive J/$\psi$, 1 map

In [ ]:
# For only 1 map

# dir_path = '~/alice/Jpsi-Jets-Analysis/workDir/PIDEfficiency/output'
# filename = 'JpsiWeightedEff25-02-09.root'
# dir_path = '~/alice/Jpsi-Jets-Analysis/workDir/PIDEfficiency/output/IdasMapsPromptNonP20Runs/LHC25b15'
dir_path = '~/alice/Jpsi-Jets-Analysis/workDir/PIDEfficiency/output/FullStatY/LHC25b16'

# dir_path = '~/alice/Jpsi-Jets-Analysis/workDir/PIDEfficiency/output/IdasMapsPromptNonPNevts10000/LHC25b15'
filename = 'AnalysisResults_TrackBarrel_Conversions_withPID_nSigmaEl-4-4.root'
path = f"{dir_path}/{filename}"
outDir = "output/PIDefficiency/cutOnY"

In [ ]:
reload(hutils)
graphPIDEff25b14, legPIDEff25b14 = hutils.PIDEfficiencyPlot(
    a="hWeightedJpsiEffPtY",
    b="hMatchedJpsiPtY",
    path=path,
    graph_title="J/#psi PID Efficiency",
    legText="LHC25b14",
    ptBins = np.concatenate((np.arange(0, 10, 1), np.arange(10, 22, 2)), axis=0)
)

In [ ]:
cPIDEff25b14 = ROOT.TCanvas()
graphPIDEff25b14.SetTitle("J/#psi PID Efficiency, n#sigma_{El} = 4") #Change previous title
graphPIDEff25b14.Draw("AP")
legPIDEff25b14.Draw()
cPIDEff25b14.Draw()

---
# Inclusive J/$\psi$, all maps

In [ ]:
allDatasetsEfficiencies = [] # list of lists of (graph, label) tuples, one list per dataset
allDatasetsAvgEfficiencies = []

palette = ROOT.kDarkRainBow

# ROOT.kSunset
# ROOT.kViridis
# ROOT.kRust
# ROOT.kPlum
# ROOT.kDarkRainBow
# ROOT.kBird

# MCDatasets = ["LHC25b14", "LHC25b15", "LHC25b16", "LHC25b17"]
# MCDatasets = ["LHC25b14", "LHC25b15"]
MCDatasets = ["LHC25b16", "LHC25b17"]
promptMCs = ["LHC25b14", "LHC25b16"]
nonPromptMCs = ["LHC25b15", "LHC25b17"]
mapsUpperDirPath = "~/alice/Jpsi-Jets-Analysis/workDir/PIDEfficiency/output/FullStatY"  # Expected subdirs in it with names in MCDatasets

In [ ]:
# All Maps
reload(hutils)
ptBinsGeom = np.round(np.geomspace(1.0, 30.0, 30), 1)

canvasesSingleDatasetEfficiencies = []
legsAliceInclusive = []
legsBodyInclusive = []
for MCDataset in MCDatasets:
    mapsDirPath = f"{mapsUpperDirPath}/{MCDataset}"
    singleDatasetEfficiencies, legAlice = hutils.PIDEfficiencyManyMaps(mapsDirPath, MCDataset, ptBins=ptBinsGeom)
    c = ROOT.TCanvas()
    canvasesSingleDatasetEfficiencies.append(c)
    legBody = ROOT.TLegend(0.55, 0.5, 0.9, 0.8)
    legBody.SetFillStyle(0)
    legBody.SetBorderSize(0)
    legBody.SetTextSize(0.02)
    legsBodyInclusive.append(legBody)
    legsAliceInclusive.append(legAlice)
    ROOT.gStyle.SetPalette(palette)
    nGraphs = len(singleDatasetEfficiencies)
    nColors = ROOT.gStyle.GetNumberOfColors()
    for i, (graph, label) in enumerate(singleDatasetEfficiencies):
        NexcludedUpperColor = 30 # Important for example for kSunset, to exclude very white colors. ~ 30
        idColor = int(i * (nColors - 1 - NexcludedUpperColor) / (nGraphs - 1))
        color = ROOT.gStyle.GetColorPalette(idColor)
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)
        
        if i == 0:
            graph.SetTitle(f"#splitline{{{MCDataset}}}{{J/#psi PID Efficiency}}")
            graph.Draw("AP")
        else:
            graph.Draw("P SAME")
        
        legBody.AddEntry(graph, label, "p")
    legBody.SetHeader("PID Cuts", "C")
    legBody.Draw()
    legAlice.Draw()
    pdf_name = f"PIDEfficiency_AllMaps{MCDataset}.pdf"
    c.Print(f"{outDir}/{pdf_name}")

    # Closes PDF
    c.Print(f"{outDir}/{pdf_name}" + "]")

    print(f"Saved {outDir}/{pdf_name}")
    allDatasetsEfficiencies.append(singleDatasetEfficiencies)
    singleDatasetAvgEff = hutils.averagePIDEfficiency(singleDatasetEfficiencies)
    allDatasetsAvgEfficiencies.append(singleDatasetAvgEff)

In [ ]:
# Log X
ROOT.gPad.SetLogx()
c.Draw()
pdf_name = f"PIDEfficiency_AllMaps{MCDataset}LogX.pdf"
c.Print(f"{outDir}/{pdf_name}")

In [ ]:
# Average PID Efficiency for all maps for all MCDatasets (e.g. LHC25b14)
reload(hutils)
hutils.test_utils()
cAvgEff = ROOT.TCanvas()
legend = ROOT.TLegend(0.6, 0.7, 0.88, 0.88)
colors = [ROOT.kRed-2, ROOT.kOrange-2, ROOT.kGreen-1, ROOT.kBlue-2]
for i, (singleDatasetAvgEff, MCDataset) in enumerate(zip(allDatasetsAvgEfficiencies, MCDatasets)):
    color = colors[i]
    singleDatasetAvgEff.SetLineColor(color)
    singleDatasetAvgEff.SetMarkerColor(color)
    singleDatasetAvgEff.SetMarkerStyle(20 + i)
    singleDatasetAvgEff.SetTitle("J/#psi Avg. PID-Efficiency between datasets")
    if i == 0:
        singleDatasetAvgEff.Draw("AP")
    else:
        singleDatasetAvgEff.Draw("P SAME")
    legend.AddEntry(singleDatasetAvgEff, MCDataset, "lp")
legend.Draw()
cAvgEff.Update()
pdf_name = "PIDEfficiency_Average_AllMaps.pdf"
cAvgEff.Print(f"{outDir}/{pdf_name}")

---
# Prompt and non-prompt J/$\psi$

In [ ]:
# For prompt, only 1 map and dataset

# reload(hutils)
# dir_path = '~/alice/Jpsi-Jets-Analysis/workDir/PIDEfficiency/output/IdasMapsPromptNonP20Runs/LHC25b14'
# filename = 'AnalysisResults_TrackBarrel_Conversions_withPID_nSigmaEl-2-2.root'
# path = f"{dir_path}/{filename}"

# reload(hutils)
# graphPromptPIDEff25b14, legPromptPIDEff25b14 = hutils.PIDEfficiencyPlot(
#     a="hWeightedPromptJpsiEffPtY",
#     b="hMatchedPromptJpsiPtY",
#     path=path,
#     graph_title="Prompt J/#psi PID Efficiency",
#     legText="LHC25b14",
#     ptBins = np.concatenate((np.arange(0, 10, 1), np.arange(10, 22, 2)), axis=0)
# )

# cPromptPIDEff25b14 = ROOT.TCanvas()
# graphPromptPIDEff25b14.SetTitle("Prompt J/#psi PID Efficiency, n#sigma_{El} = 4") #Change previous title
# graphPromptPIDEff25b14.Draw("AP")
# legPromptPIDEff25b14.Draw()
# cPromptPIDEff25b14.Draw()

## Prompt

In [ ]:
allDatasetsPromptEfficiencies = [] # list of lists of (graph, label) tuples, one list per dataset
allDatasetsAvgPromptEfficiencies = []

# All Maps,
reload(hutils)
hutils.test_utils()

ptBinsGeom = np.round(np.geomspace(1.0, 35.0, 35), 1)
ptBinsLowStat = np.round(np.geomspace(1.0, 8.0, 8), 1)
ptBinsLowStat = np.concatenate((ptBinsLowStat, np.arange(10, 35, 5)))
ptBinsLowStat = np.round(ptBinsLowStat, 1)

canvasesSingleDatasetPromptEfficiencies = []
legsAlicePrompt = []
legsBodyPrompt = []
for MCDataset in MCDatasets:
    mapsDirPath = f"{mapsUpperDirPath}/{MCDataset}"
    if MCDataset in promptMCs:
        usedPtBins = ptBinsGeom
    else:
        usedPtBins = ptBinsLowStat
    singleDatasetPromptEfficiencies, legAlice = hutils.PIDEfficiencyManyMaps(mapsDirPath, MCDataset, weightedHist="hWeightedPromptJpsiEffPtY", matchedHist="hMatchedPromptJpsiPtY", ptBins=usedPtBins)
    c = ROOT.TCanvas()
    c.cd()
    canvasesSingleDatasetPromptEfficiencies.append(c)
    legsAlicePrompt.append(legAlice)
    legBody = ROOT.TLegend(0.55, 0.5, 0.9, 0.8)
    legBody.SetFillStyle(0)
    legBody.SetBorderSize(0)
    legBody.SetTextSize(0.02)
    legsBodyPrompt.append(legBody)
    ROOT.gStyle.SetPalette(palette)
    nGraphs = len(singleDatasetPromptEfficiencies)
    nColors = ROOT.gStyle.GetNumberOfColors()
    for i, (graph, label) in enumerate(singleDatasetPromptEfficiencies):
        # ROOT.SetOwnership(graph, False)
        graph = graph.Clone()
        NexcludedUpperColor = 30 # Important for example for kSunset, to exclude very white colors. ~ 30
        idColor = int(i * (nColors - 1 - NexcludedUpperColor) / (nGraphs - 1))
        color = ROOT.gStyle.GetColorPalette(idColor)
        # markerStyle = 20 + i
        # graph.SetMarkerStyle(markerStyle)
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)
        
        if i == 0:
            graph.SetTitle(f"Prompt-J/#psi PID Efficiency") #, {MCDataset}")
            graph.Draw("AP")
        else:
            graph.Draw("P SAME")
        
        legBody.AddEntry(graph, label, "p")
    legBody.SetHeader("PID Cuts", "C")
    legAlice.Draw()
    legBody.Draw()
    pdf_name = f"PromptPIDEfficiency_AllMaps{MCDataset}.pdf"
    c.Print(f"{outDir}/{pdf_name}")

    # Closes PDF
    c.Print(f"{outDir}/{pdf_name}" + "]")

    print(f"Saved {outDir}/{pdf_name}")
    allDatasetsPromptEfficiencies.append(singleDatasetPromptEfficiencies)
    singleDatasetAvgPromptEff = hutils.averagePIDEfficiency(singleDatasetPromptEfficiencies)
    allDatasetsAvgPromptEfficiencies.append(singleDatasetAvgPromptEff)

In [ ]:
# Average PID Efficiency for all maps for all MCDatasets (e.g. LHC25b14)
reload(hutils)
hutils.test_utils()
cAvgPromptEff = ROOT.TCanvas()
legend = ROOT.TLegend(0.6, 0.7, 0.88, 0.88)
colors = [ROOT.kRed-2, ROOT.kOrange-2, ROOT.kGreen-1, ROOT.kBlue-2]
plotOnlyPromptDatasets = False # If True, only plots LHC25b14 and LHC25b16, which are the prompt datasets. If False, plots all datasets, but only the prompt PID efficiency.
isFirstPlot = True
for i, (singleDatasetAvgPromptEff, MCDataset) in enumerate(zip(allDatasetsAvgPromptEfficiencies, MCDatasets)):
    if (plotOnlyPromptDatasets and MCDataset in ["LHC25b14", "LHC25b16"]) or not plotOnlyPromptDatasets:
        color = colors[i]
        singleDatasetAvgPromptEff.SetLineColor(color)
        singleDatasetAvgPromptEff.SetMarkerColor(color)
        singleDatasetAvgPromptEff.SetMarkerStyle(20 + i)
        if isFirstPlot:
            singleDatasetAvgPromptEff.SetTitle("Prompt-J/#psi Avg. PID-Efficiency")
            singleDatasetAvgPromptEff.Draw("AP")
            isFirstPlot = False
        else:
            singleDatasetAvgPromptEff.Draw("P SAME")
        legend.AddEntry(singleDatasetAvgPromptEff, MCDataset, "lp")
legend.Draw()
cAvgPromptEff.Update()
pdf_name = "PromptPIDEfficiency_Average_AllMaps.pdf"
cAvgPromptEff.Print(f"{outDir}/{pdf_name}")

## Non-prompt

In [ ]:
allDatasetsNonPromptEfficiencies = [] # list of lists of (graph, label) tuples, one list per dataset
allDatasetsAvgNonPromptEfficiencies = []

# All Maps,
# reload(hutils)
# hutils.test_utils()
ptBinsGeom = np.round(np.geomspace(1.0, 40.0, 80), 1)

canvasesSingleDatasetNonPromptEfficiencies = []
legsAliceNonPrompt = []
legsBodyNonPrompt = []
for MCDataset in MCDatasets:
    mapsDirPath = f"{mapsUpperDirPath}/{MCDataset}"
    print(mapsDirPath)
    singleDatasetNonPromptEfficiencies, legAlice = hutils.PIDEfficiencyManyMaps(mapsDirPath, MCDataset, weightedHist="hWeightedNonPromptJpsiEffPtY", matchedHist="hMatchedNonPromptJpsiPtY", ptBins=ptBinsGeom)
    print()
    c = ROOT.TCanvas()
    canvasesSingleDatasetNonPromptEfficiencies.append(c)
    # legend = ROOT.TLegend(0.6, 0.2, 0.88, 0.5)
    # legend.SetHeader("PID Cuts", "C")
    legBody = ROOT.TLegend(0.55, 0.5, 0.9, 0.8)
    legBody.SetFillStyle(0)
    legBody.SetBorderSize(0)
    legBody.SetTextSize(0.02)
    legsAliceNonPrompt.append(legAlice)
    legsBodyNonPrompt.append(legBody)
    ROOT.gStyle.SetPalette(palette)
    nGraphs = len(singleDatasetNonPromptEfficiencies)
    nColors = ROOT.gStyle.GetNumberOfColors()
    for i, (graph, label) in enumerate(singleDatasetNonPromptEfficiencies):
        NexcludedUpperColor = 30 # Important for example for kSunset, to exclude very white colors. ~ 30
        idColor = int(i * (nColors - 1 - NexcludedUpperColor) / (nGraphs - 1))
        color = ROOT.gStyle.GetColorPalette(idColor)
        graph.SetMarkerColor(color)
        graph.SetLineColor(color)
        if i == 0:
            graph.SetTitle(f"Non-Prompt-J/#psi PID Efficiency") #, {MCDataset}")
            graph.Draw("AP")
        else:
            graph.Draw("P SAME")
        legBody.AddEntry(graph, label, "p")
    legBody.SetHeader("PID Cuts", "C")
    legAlice.Draw()
    legBody.Draw()
    pdf_name = f"NonPromptPIDEfficiency_AllMaps{MCDataset}.pdf"
    c.Print(f"{outDir}/{pdf_name}")

    # Closes PDF
    c.Print(f"{outDir}/{pdf_name}" + "]")

    print(f"Saved {outDir}/{pdf_name}")
    allDatasetsNonPromptEfficiencies.append(singleDatasetNonPromptEfficiencies)
    singleDatasetAvgNonPromptEff = hutils.averagePIDEfficiency(singleDatasetNonPromptEfficiencies)
    allDatasetsAvgNonPromptEfficiencies.append(singleDatasetAvgNonPromptEff)

In [ ]:
# Average PID Efficiency for all maps for all MCDatasets (e.g. LHC25b14)
cAvgNonPromptEff = ROOT.TCanvas()
legend = ROOT.TLegend(0.6, 0.7, 0.88, 0.88)
colors = [ROOT.kRed-2, ROOT.kOrange-2, ROOT.kGreen-1, ROOT.kBlue-2]
plotOnlyNonPromptDatasets = False
isFirstPlot = True
for i, (singleDatasetAvgNonPromptEff, MCDataset) in enumerate(zip(allDatasetsAvgNonPromptEfficiencies, MCDatasets)):
    if (plotOnlyNonPromptDatasets and MCDataset in ["LHC25b15", "LHC25b17"]) or not plotOnlyNonPromptDatasets:
        color = colors[i]
        singleDatasetAvgNonPromptEff.SetLineColor(color)
        singleDatasetAvgNonPromptEff.SetMarkerColor(color)
        singleDatasetAvgNonPromptEff.SetMarkerStyle(20 + i)
        if isFirstPlot:
            singleDatasetAvgNonPromptEff.SetTitle("Non-Prompt-J/#psi Avg. PID-Efficiency")
            singleDatasetAvgNonPromptEff.Draw("AP")
            isFirstPlot = False
        else:
            singleDatasetAvgNonPromptEff.Draw("P SAME")
        legend.AddEntry(singleDatasetAvgNonPromptEff, MCDataset, "lp")
legend.Draw()
cAvgNonPromptEff.Update()
pdf_name = "NonPromptPIDEfficiency_Average_AllMaps.pdf"
cAvgNonPromptEff.Print(f"{outDir}/{pdf_name}")